<h2>Apply complex domain-specific feature transformations that align with business context, including ratio features, custom aggregations, and interaction terms based on in-depth exploratory data analysis and subject matter expertise.</h2>

In [2]:
import pandas as pd
import numpy as np

In [3]:
np.random.seed(42)
n_samples = 500

In [4]:
df = pd.DataFrame({
    'customer_id': np.arange(1001, 1001 + n_samples),
    'monthly_income': np.random.uniform(2500, 15000, size=n_samples),
    'monthly_debt_payments': np.random.uniform(500, 5000, size=n_samples),
    'revolving_credit_limit': np.random.uniform(5000, 50000, size=n_samples),
    'revolving_credit_balance': np.random.uniform(1000, 45000, size=n_samples),
    'credit_inquiries_last_6m': np.random.randint(0, 8, size=n_samples),
    'total_successful_payments': np.random.randint(10, 60, size=n_samples),
    'total_due_payments': np.random.randint(12, 60, size=n_samples),
    'risk_tier': np.random.choice(['Low', 'Medium', 'High'], size=n_samples)
})

In [5]:
print("Raw Financial Dataset Sample:")
df.head()

Raw Financial Dataset Sample:


,customer_id,monthly_income,monthly_debt_payments,revolving_credit_limit,revolving_credit_balance,credit_inquiries_last_6m,total_successful_payments,total_due_payments,risk_tier
0,1001,7181.751486,3641.727713,13330.981798,23839.598546,3,18,34,High
1,1002,14383.928830,2912.433649,29385.542632,22084.002616,6,18,25,Low
2,1003,11649.924273,1892.874273,44282.562614,2128.250895,2,13,38,Medium
3,1004,9983.231052,4162.077589,37950.119888,16014.904416,6,44,40,Low
4,1005,4450.233006,3581.290276,41295.251654,17728.607227,3,26,50,Medium


In [6]:
df['dti_ratio'] = df['monthly_debt_payments'] / (df['monthly_income'] + 1e-5)

In [7]:
df['revolving_utilization_ratio'] = df['revolving_credit_balance'] / (df['revolving_credit_limit'] + 1e-5)

In [8]:
df['disposable_income'] = df['monthly_income'] - df['monthly_debt_payments']

In [9]:
print("Calculated Ratio Features:")
df[['customer_id', 'dti_ratio', 'revolving_utilization_ratio', 'disposable_income']].head()

Calculated Ratio Features:


,customer_id,dti_ratio,revolving_utilization_ratio,disposable_income
0,1001,0.507081,1.788285,3540.023773
1,1002,0.202478,0.751526,11471.495182
2,1003,0.162480,0.048061,9757.049999
3,1004,0.416907,0.421999,5821.153464
4,1005,0.804742,0.429313,868.942729


In [15]:
if 'dti_ratio' not in df.columns:
    df['dti_ratio'] = df['monthly_debt_payments'] / (df['monthly_income'] + 1e-5)
if 'revolving_utilization_ratio' not in df.columns:
    df['revolving_utilization_ratio'] = df['revolving_credit_balance'] / (df['revolving_credit_limit'] + 1e-5)

In [18]:
df['risk_tier_mean_dti'] = df.groupby('risk_tier')['dti_ratio'].transform('mean')

In [19]:
df['risk_tier_mean_utilization'] = df.groupby('risk_tier')['revolving_utilization_ratio'].transform('mean')

In [20]:
df['dti_vs_risk_cohort'] = df['dti_ratio'] - df['risk_tier_mean_dti']

In [21]:
df['utilization_vs_risk_cohort'] = df['revolving_utilization_ratio'] - df['risk_tier_mean_utilization']

In [22]:
print("Cohort-Relative Features Created Successfully:")
print(df[['customer_id', 'risk_tier', 'risk_tier_mean_dti', 'dti_vs_risk_cohort']].head())

Cohort-Relative Features Created Successfully:
   customer_id risk_tier  risk_tier_mean_dti  dti_vs_risk_cohort
0         1001      High            0.404402            0.102679
1         1002       Low            0.391079           -0.188601
2         1003    Medium            0.370797           -0.208318
3         1004       Low            0.391079            0.025828
4         1005    Medium            0.370797            0.433945


In [23]:
df['payment_reliability_rate'] = df['total_successful_payments'] / (df['total_due_payments'] + 1e-5)

In [24]:
df['inquiry_utilization_stress'] = df['credit_inquiries_last_6m'] * df['revolving_utilization_ratio']

In [25]:
df['solvency_weighted_reliability'] = df['payment_reliability_rate'] * np.log1p(np.maximum(0, df['disposable_income']))

In [26]:
print("Final Advanced Interaction Features:")
df[['customer_id', 'payment_reliability_rate', 'inquiry_utilization_stress', 'solvency_weighted_reliability']].head()

Final Advanced Interaction Features:


,customer_id,payment_reliability_rate,inquiry_utilization_stress,solvency_weighted_reliability
0,1001,0.529412,5.364856,4.326442
1,1002,0.720000,4.509157,6.730347
2,1003,0.342105,0.096121,3.142526
3,1004,1.100000,2.531993,9.536366
4,1005,0.520000,1.287940,3.519582
